In [ ]:
import torch
from torch import nn,optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

data

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    transform=transform,
    download=True,
)
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0,
)

model

In [ ]:
class Generator(nn.Module):
    def __init__(self, noise_dim=100):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(noise_dim, 256),
            nn.LeakyReLU(0.2),

            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),

            nn.Linear(512, 28 * 28),

            # 输出范围与真实图片保持一致：[-1, 1]
            nn.Tanh(),
        )

    def forward(self, z):
        images = self.network(z)

        # [batch_size, 784] -> [batch_size, 1, 28, 28]
        return images.view(-1, 1, 28, 28)
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.LeakyReLU(0.2),

            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),

            nn.Linear(256, 1),

            # 将输出转换到 [0, 1]
            nn.Sigmoid(),
        )

    def forward(self, images):
        # [batch_size, 1, 28, 28] -> [batch_size, 784]
        images = images.flatten(start_dim=1)

        return self.network(images)
criterion = nn.BCELoss()
def discriminator_loss(
    discriminator,
    real_images,
    fake_images,
):
    batch_size = real_images.size(0)

    real_labels = torch.ones(
        batch_size, 1, device=real_images.device
    )
    fake_labels = torch.zeros(
        batch_size, 1, device=real_images.device
    )

    # 判断真实图片
    real_predictions = discriminator(real_images)
    real_loss = criterion(real_predictions, real_labels)

    # 判断生成图片
    # detach 防止训练判别器时更新生成器
    fake_predictions = discriminator(fake_images.detach())
    fake_loss = criterion(fake_predictions, fake_labels)

    total_loss = real_loss + fake_loss

    return total_loss, real_loss, fake_loss
def generator_loss(discriminator, fake_images):
    batch_size = fake_images.size(0)

    # 生成器希望判别器把假图片判断成真图片
    target_labels = torch.ones(
        batch_size, 1, device=fake_images.device
    )

    predictions = discriminator(fake_images)
    loss = criterion(predictions, target_labels)

    return loss

train

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
noise_dim = 100
learning_rate = 2e-4
#定义模型
generator = Generator(noise_dim=noise_dim).to(device)
discriminator = Discriminator().to(device)
#定义优化器
generator_optimizer = optim.Adam(
    generator.parameters(),
    lr=learning_rate,
    betas=(0.5, 0.999),
)
discriminator_optimizer = optim.Adam(
    discriminator.parameters(),
    lr=learning_rate,
    betas=(0.5, 0.999),
)
#训练函数
def train(
    generator,
    discriminator,
    train_loader,
    generator_optimizer,
    discriminator_optimizer,
    device,
    noise_dim,
    epoch,
):
    generator.train()
    discriminator.train()

    total_generator_loss = 0
    total_discriminator_loss = 0

    for batch_index, (real_images, _) in enumerate(train_loader):
        real_images = real_images.to(device)
        batch_size = real_images.size(0)

        # ---------------------------------
        # 1. 训练判别器
        # ---------------------------------
        discriminator_optimizer.zero_grad()

        noise = torch.randn(
            batch_size,
            noise_dim,
            device=device,
        )
        fake_images = generator(noise).detach()

        d_loss, d_real_loss, d_fake_loss = discriminator_loss(
            discriminator,
            real_images,
            fake_images,
        )

        d_loss.backward()
        discriminator_optimizer.step()

        # ---------------------------------
        # 2. 训练生成器
        # ---------------------------------
        generator_optimizer.zero_grad()

        noise = torch.randn(
            batch_size,
            noise_dim,
            device=device,
        )
        fake_images = generator(noise)

        g_loss = generator_loss(
            discriminator,
            fake_images,
        )

        g_loss.backward()
        generator_optimizer.step()

        total_discriminator_loss += d_loss.item()
        total_generator_loss += g_loss.item()



    number_of_batches = len(train_loader)

    print(
        f"Epoch {epoch} 完成 | "
        f"平均 D Loss: "
        f"{total_discriminator_loss / number_of_batches:.4f} | "
        f"平均 G Loss: "
        f"{total_generator_loss / number_of_batches:.4f}"
    )
#main函数
if __name__ == "__main__":
    epochs = 20

    for epoch in range(1, epochs + 1):
        train(
            generator=generator,
            discriminator=discriminator,
            train_loader=train_loader,
            generator_optimizer=generator_optimizer,
            discriminator_optimizer=discriminator_optimizer,
            device=device,
            noise_dim=noise_dim,
            epoch=epoch,
        )

test

In [ ]:
generator.eval()
num_samples = 16
noise_dim = 100
with torch.no_grad():
    # 随机采样噪声
    noise = torch.randn(
        num_samples,
        noise_dim,
        device=device,
    )

    # 生成图片，形状为 [16, 1, 28, 28]
    generated_images = generator(noise).cpu()

    # 将像素范围从 [-1, 1] 恢复到 [0, 1]
    generated_images = (generated_images + 1) / 2
fig, axes = plt.subplots(4, 4, figsize=(6, 6))
for image, ax in zip(generated_images, axes.flat):
    ax.imshow(
        image.squeeze(0),
        cmap="gray",
        vmin=0,
        vmax=1,
    )
    ax.axis("off")
plt.tight_layout()
plt.show()